## F1 Silver Layer Pipeline
**Sources:** `drivers.json`, `results.json` from landing zone  
**Target:** `f1_catalognew.f1_schema.drivers_silver`, `f1_catalognew.f1_schema.results_silver`

**Transformations applied:**
- Replace `\N` markers with proper nulls
- Flatten nested structs (drivers `name`)
- Cast strings to correct types (dates, integers, doubles)
- Rename columns to `snake_case` convention
- Drop raw/unnecessary columns (`url`, raw `time`)
- Add `ingestion_timestamp` audit column

In [0]:
df = spark.read.json(
    "/Volumes/f1_catalognew/f1_schema/landing_zone/drivers.json/"
)
display(df)

In [0]:
from pyspark.sql.functions import col, when, lit, current_timestamp, concat_ws
from pyspark.sql.types import IntegerType, DateType, DoubleType, LongType

df_drivers_silver = (
    df
    # Flatten nested name struct
    .withColumn("forename", col("name.forename"))
    .withColumn("surname", col("name.surname"))
    .withColumn("full_name", concat_ws(" ", col("name.forename"), col("name.surname")))
    .drop("name")
    # Replace '\N' markers with proper nulls
    .withColumn("code", when(col("code") == "\\N", lit(None)).otherwise(col("code")))
    .withColumn("number", when(col("number") == "\\N", lit(None)).otherwise(col("number")))
    # Cast to correct types
    .withColumn("number", col("number").cast(IntegerType()))
    .withColumn("dob", col("dob").cast(DateType()))
    # Rename to snake_case
    .withColumnRenamed("driverId", "driver_id")
    .withColumnRenamed("driverRef", "driver_ref")
    .withColumnRenamed("dob", "date_of_birth")
    # Drop unnecessary columns
    .drop("url")
    # Add audit metadata
    .withColumn("ingestion_timestamp", current_timestamp())
    # Final column order
    .select(
        "driver_id", "driver_ref", "code", "number",
        "forename", "surname", "full_name",
        "date_of_birth", "nationality", "ingestion_timestamp"
    )
)

display(df_drivers_silver)

In [0]:
silver_drivers_table = "f1_catalognew.silver_schema.drivers_silver"

df_drivers_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_drivers_table)

print(f"✓ Written: {silver_drivers_table}")

In [0]:
df_results_raw = spark.read.json(
    "/Volumes/f1_catalognew/f1_schema/landing_zone/results.json/"
)
print(f"Results bronze: {df_results_raw.count()} rows")
display(df_results_raw)

In [0]:
# Columns with '\N' null markers that need cleaning + type casting
null_marker_cols = [
    "fastestLap", "fastestLapSpeed", "fastestLapTime",
    "milliseconds", "number", "position", "rank", "time"
]

# Start with raw data, replace \N with null across all affected columns
df_results_clean = df_results_raw
for c in null_marker_cols:
    df_results_clean = df_results_clean.withColumn(
        c, when(col(c) == "\\N", lit(None)).otherwise(col(c))
    )

df_results_silver = (
    df_results_clean
    # Cast to correct types
    .withColumn("fastest_lap", col("fastestLap").cast(IntegerType()))
    .withColumn("fastest_lap_speed", col("fastestLapSpeed").cast(DoubleType()))
    .withColumn("fastest_lap_time", col("fastestLapTime"))  # keep as string (mm:ss.sss)
    .withColumn("milliseconds", col("milliseconds").cast(LongType()))
    .withColumn("number", col("number").cast(IntegerType()))
    .withColumn("position", col("position").cast(IntegerType()))
    .withColumn("rank", col("rank").cast(IntegerType()))
    .withColumn("time", col("time"))  # keep as string (mixed format: absolute + gap)
    # Rename remaining columns to snake_case
    .withColumnRenamed("resultId", "result_id")
    .withColumnRenamed("raceId", "race_id")
    .withColumnRenamed("driverId", "driver_id")
    .withColumnRenamed("constructorId", "constructor_id")
    .withColumnRenamed("positionOrder", "position_order")
    .withColumnRenamed("positionText", "position_text")
    .withColumnRenamed("statusId", "status_id")
    # Drop original camelCase columns replaced by snake_case versions
    .drop("fastestLap", "fastestLapSpeed", "fastestLapTime")
    # Add audit metadata
    .withColumn("ingestion_timestamp", current_timestamp())
    # Final column order
    .select(
        "result_id", "race_id", "driver_id", "constructor_id",
        "number", "grid", "position", "position_text", "position_order",
        "points", "laps", "time", "milliseconds",
        "fastest_lap", "fastest_lap_time", "fastest_lap_speed",
        "rank", "status_id", "ingestion_timestamp"
    )
)

display(df_results_silver)

In [0]:
silver_results_table = "f1_catalognew.silver_schema.results_silver"

df_results_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_results_table)

print(f"✓ Written: {silver_results_table}")

In [0]:
from pyspark.sql.functions import count, sum as _sum, isnull

# --- Drivers Silver ---
df_drv = spark.table("f1_catalognew.silver_schema.drivers_silver")
drv_count = df_drv.count()

drv_nulls = df_drv.select(
    _sum(isnull("code").cast("int")).alias("null_codes"),
    _sum(isnull("number").cast("int")).alias("null_numbers"),
    _sum(isnull("date_of_birth").cast("int")).alias("null_dob"),
).first()

print(f"=== drivers_silver ===")
print(f"Rows: {drv_count} | Null codes: {drv_nulls['null_codes']} | Null numbers: {drv_nulls['null_numbers']} | Null DOB: {drv_nulls['null_dob']}")
df_drv.printSchema()

# --- Results Silver ---
df_res = spark.table("f1_catalognew.silver_schema.results_silver")
res_count = df_res.count()

res_nulls = df_res.select(
    _sum(isnull("position").cast("int")).alias("null_positions"),
    _sum(isnull("fastest_lap").cast("int")).alias("null_fastest_lap"),
    _sum(isnull("milliseconds").cast("int")).alias("null_ms"),
).first()

print(f"\n=== results_silver ===")
print(f"Rows: {res_count} | Null positions: {res_nulls['null_positions']} | Null fastest_lap: {res_nulls['null_fastest_lap']} | Null ms: {res_nulls['null_ms']}")
df_res.printSchema()